# B2.7 · Dynamic exploitation (DAST)

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**.

| | |
|---|---|
| Tools used | OWASP ZAP, Nuclei, GLM-4.6, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Turn static findings into executable probes against the replica and separate confirmed from unconfirmed.

**Why a security engineer needs it.** A SAST finding is a hypothesis, and hypotheses get argued about instead of fixed. The control it builds is: stage 12: generate and run an actual exploit against the sandbox, so the finding is confirmed or dropped.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A finding becomes a fact the moment something other than a model says so. Driving the running application is how you get that second opinion — and the oracle you choose is what makes it worth having.

> **At CyberTravels.** A finding becomes a fact when something other than a model says so — here, a request to the replica's booking endpoint that returns another traveller's card details.

## 2 · The framework

```
   candidate finding
        |
        v
   payload --> running replica --> observed behaviour
                                        |
                             +----------v-----------+
                             |  ORACLE               |
                             |  did the state change?|
                             |  did the row appear?  |
                             +----------+-----------+
                                        v
                              confirmed | not confirmed

   the oracle is the whole value. "the model thinks so" is not one.
```

**Stage 12 — Dynamic exploitation.** The stage that converts an argument into a
fact.

Everything Phase 3 produced is a hypothesis: the code *looks* vulnerable and the
sink *appears* reachable. Hypotheses get argued about in triage meetings. An
executed exploit does not — either the probe achieved the effect or it did not.

Two things this stage produces that static analysis cannot:

- **Confirmation.** A finding that survives an exploit attempt is real,
  regardless of how the model felt about it.
- **Refutation.** A finding that fails is either not exploitable in this
  configuration or not real, and both are useful answers.

The discipline that makes it trustworthy is that the probe must assert a
**concrete effect** — rows returned that should not be, a file read outside the
root — not merely that the request did not error. "No exception" is the DAST
equivalent of a shape check, and B2.0 already established what those are worth.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Where it breaks — the probe that asserts nothing

The most common DAST bug: treating "the request succeeded" as confirmation. Both probes below hit the app and return 200-equivalents; only one of them proves anything.

## 4 · The stage, as a skill

A dynamic probe proves nothing unless its assertion can fail. The skill runs the probes against a live build with a control probe alongside, then re-runs them under a weak assertion so you can watch it flag the control too.

### The skill — [`skills/appsec/dynamic-exploitation-probe/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dynamic-exploitation-probe/SKILL.md)

```yaml
name: dynamic-exploitation-probe
description: >-
  Run probes against a live build to turn a static finding into a demonstrated
  one, with a control probe and an assertion strong enough to tell them apart.
  Use when confirming exploitability, or when a dynamic test reports everything
  as vulnerable.
allowed-tools: Read, Grep, Glob, Bash
```

# The assertion is the experiment

A dynamic probe proves nothing unless its assertion can fail. A weak assertion —
"the response contains rows", "the request did not error" — confirms the control
probe as readily as the attack, and a test that confirms everything has measured
nothing.

## When to use this

After static analysis has produced hypotheses, in a replica confirmed by an
isolation check, and before any finding is reported as exploitable.

## Procedure

**1 — State each hypothesis as a prediction.** Not "SQL injection in
`list_reports`" but "requesting one owner returns rows for more than one". A
hypothesis you cannot write as a prediction is not ready to test.

**2 — Include a control probe.** A well-formed, benign request whose result is
known. It is the thing that tells you your assertion discriminates.

**3 — Write the assertion against the prediction.** Count owners returned, not
whether rows came back. Read the file's contents, not whether a path resolved.
The difference between these is the entire value of the stage.

**4 — Run all probes, including the control, and record raw results.** Then
apply the assertion. Keep both: the raw result is what somebody re-reads when
they doubt the finding.

**5 — Demonstrate the weak assertion too.** Run the same probes under the loose
check and show it flagging the control. That comparison is what stops the next
person writing one.

**6 — Mark each hypothesis confirmed, refuted, or untested** — and never
"probably". A hypothesis whose probe could not run is untested, which is
different from refuted and much more common.

## Output contract

```json
{
  "hypotheses": [{"id": "str", "prediction": "str"}],
  "probes": [{"id": "str", "kind": "attack|control", "raw": "str"}],
  "assertions": [{"id": "str", "strong": true, "verdict": "confirmed|refuted|untested"}],
  "weak_assertion_run": {"flagged": ["str"], "includes_control": true}
}
```

## Failure modes

- **No control probe.** Nothing tells you the assertion discriminates.
- **Asserting on shape.** "Rows came back" is true for the legitimate query.
- **Recording untested as refuted.** They have different owners and different
  next steps.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dynamic-exploitation-probe/scripts/dynamic_exploitation_probe.py
SCRIPT = "skills/appsec/dynamic-exploitation-probe/scripts/dynamic_exploitation_probe.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The SQL injection probe returns rows for three owners when one was requested, and the traversal probe returns the synthetic token from outside the document root; the control probe returns a single owner and is not flagged. The weak assertion confirms all three including the control. Stage 12 marks two findings CONFIRMED and one UNVALIDATED for having no probe.

## Your turn

Look at your DAST assertions. If any of them checks only for a non-error response, it is confirming findings it has not tested — and the control probe above is how you prove that in five minutes.

---

**Next → [B2.8 · Exploit chaining](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*